## 1. Imports et paramètres

Chargement des bibliothèques et définition des constantes du projet :
périmètre opérateurs MRN, table de correspondance des modes TC (Tram, TEOR, Train, Bus, BAC).

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

GTFS_DIR   = Path("../raw/atoumod-gtfs_20260512")
OUTPUT     = Path("../data/arrets_2026.gpkg")
LAYER_NAME = "arrets_2026"

TRAM_LINES = {"Métro"}
TEOR_LINES = {"T1", "T2", "T3", "T4", "T5"}

OPERATEURS_MRN = {
    "ATOUMOD001", "ATOUMOD002", "ATOUMOD005", "ATOUMOD006",
    "ATOUMOD008", "ATOUMOD012", "ATOUMOD019", "ATOUMOD040",
    "ATOUMOD041", "ATOUMOD115",
}
MODE_FALLBACK = {"ATOUMOD002": "Train"}

In [ ]:
routes     = pd.read_csv(GTFS_DIR / "routes.txt",
                         usecols=["route_id", "route_short_name", "route_type"])
trips      = pd.read_csv(GTFS_DIR / "trips.txt",
                         usecols=["trip_id", "route_id"])
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt",
                         usecols=["trip_id", "stop_id"], low_memory=False)
stops      = pd.read_csv(GTFS_DIR / "stops.txt")

print(f"routes     : {len(routes):>6} lignes")
print(f"trips      : {len(trips):>6} lignes")
print(f"stop_times : {len(stop_times):>6} lignes")
print(f"stops      : {len(stops):>6} lignes")

## 2. Lecture des fichiers GTFS

Chargement des quatre tables nécessaires depuis le dossier GTFS ATOUMOD :
`routes.txt`, `trips.txt`, `stop_times.txt`, `stops.txt`.
La jointure entre ces tables suit la chaîne : `stops → stop_times → trips → routes`.

In [ ]:
import numpy as np

# Mode par ligne (vectorisé)
conditions = [
    routes["route_type"] == 2,
    routes["route_type"] == 4,
    routes["route_short_name"].isin(TRAM_LINES),
    routes["route_short_name"].isin(TEOR_LINES),
]
choix = ["Train", "BAC", "Tram", "TEOR"]
routes["mode"] = np.select(conditions, choix, default="Bus")

# Jointure vectorisée
merged = (stop_times
    .merge(trips, on="trip_id")
    .merge(routes[["route_id", "route_short_name", "mode"]], on="route_id")
)

# Dédoublonnage et agrégation (stop_id × mode) → lignes
idx = (merged[["stop_id", "mode", "route_short_name"]]
    .drop_duplicates()
    .groupby(["stop_id", "mode"])["route_short_name"]
    .agg(lambda x: ", ".join(sorted(x)))
    .reset_index()
    .rename(columns={"route_short_name": "lignes"})
)
print(f"Combinaisons (arrêt × mode) : {len(idx)}")
idx.head(10)

## 3. Jointure et attribution du mode

Jointure vectorisée `stops → stop_times → trips → routes` pour relier chaque arrêt
à ses lignes et à son mode TC. Le mode est attribué par `np.select` sur
`route_type` et `route_short_name` selon la table de correspondance ATOUMOD001.

Résultat : un index `(stop_id × mode)` avec la liste des lignes desservant
chaque arrêt pour chaque mode.

In [ ]:
stops["operateur"] = stops["stop_id"].str.split(":").str[-1]
stops_76 = stops[
    stops["stop_id"].str.startswith("FR:76") &
    stops["operateur"].isin(OPERATEURS_MRN)
].copy()

rows = []

# Arrêts avec passages — une ligne par (stop_id × mode)
avec = stops_76.merge(idx, on="stop_id", how="inner")
for _, r in avec.iterrows():
    rows.append({"nom": r["stop_name"], "horizon": "2026",
                 "mode": r["mode"], "lignes": r["lignes"],
                 "operateur": r["operateur"], "id_source": r["stop_id"],
                 "geometry": Point(r["stop_lon"], r["stop_lat"])})

# Arrêts sans passage — fallback mode
sans = stops_76[~stops_76["stop_id"].isin(idx["stop_id"])]
for _, r in sans.iterrows():
    rows.append({"nom": r["stop_name"], "horizon": "2026",
                 "mode": MODE_FALLBACK.get(r["operateur"], "Bus"),
                 "lignes": "", "operateur": r["operateur"],
                 "id_source": r["stop_id"],
                 "geometry": Point(r["stop_lon"], r["stop_lat"])})

gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326").to_crs("EPSG:2154")
print(f"Lignes totales (arrêts × modes) : {len(gdf)}")


## 4. Construction de la couche géographique

Filtre sur les arrêts Seine-Maritime (`FR:76`) et les opérateurs MRN retenus.
Chaque arrêt est dupliqué autant de fois qu'il y a de modes distincts qui
le desservent. Les 496 arrêts sans passage dans `stop_times` sont conservés
avec un mode déduit de l'opérateur (fallback).

Projection finale en **EPSG:2154** (Lambert-93).

In [ ]:
import numpy as np

# Mode par ligne (vectorisé)
conditions = [
    routes["route_type"] == 2,
    routes["route_type"] == 4,
    routes["route_short_name"].isin(TRAM_LINES),
    routes["route_short_name"].isin(TEOR_LINES),
]
choix = ["Train", "BAC", "Tram", "TEOR"]
routes["mode"] = np.select(conditions, choix, default="Bus")

# Jointure vectorisée
merged = (stop_times
    .merge(trips, on="trip_id")
    .merge(routes[["route_id", "route_short_name", "mode"]], on="route_id")
)

# Dédoublonnage et agrégation (stop_id × mode) → lignes
idx = (merged[["stop_id", "mode", "route_short_name"]]
    .drop_duplicates()
    .groupby(["stop_id", "mode"])["route_short_name"]
    .agg(lambda x: ", ".join(sorted(x)))
    .reset_index()
    .rename(columns={"route_short_name": "lignes"})
)
print(f"Combinaisons (arrêt × mode) : {len(idx)}")
idx.head(10)

> **Note** — Les arrêts sans lignes renseignées (`lignes = ""`) correspondent
> aux arrêts du réseau TAEx (dessertes péri-urbaines ATOUMOD001) absents de
> `stop_times`, probablement hors période calendaire du GTFS. Ils sont conservés
> pour ne pas perdre de points d'accès potentiels.

## 5. Contrôle qualité

Vérification de la répartition par mode et par opérateur.
Valeurs attendues : ~60 arrêts Tram, ~150 TEOR, ~50 Train, ~17 BAC, ~4 400 Bus.

In [ ]:
# Contrôle rapide à ajouter en cellule 5
print("Répartition par mode :")
display(gdf["mode"].value_counts().to_frame())

print("\nArrêts Tram :")
display(gdf[gdf["mode"] == "Tram"][["nom", "lignes", "operateur"]].head(10))

print("\nArrêts TEOR :")
display(gdf[gdf["mode"] == "TEOR"][["nom", "lignes", "operateur"]].head(10))

print("\nArrêts sans lignes (fallback) :")
display(gdf[gdf["lignes"] == ""][["nom", "mode", "operateur"]].head(10))

print("\nRépartition par opérateur :")
display(gdf["operateur"].value_counts().to_frame())

print(f"\nCRS : {gdf.crs}")
gdf.head(5)

> Les doublons par sens (ex. *Georges Braque ×2*) sont normaux : chaque quai
> directionnel est un `stop_id` distinct dans le GTFS (aller / retour).
> Cette granularité est utile pour les calculs d'isochrones.

## 6. Export

Export de la couche en **GeoPackage** (`arrets_2026.gpkg`, couche `arrets_2026`)
dans le dossier `data/`, aux côtés des graphes OSM piéton et cyclable.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
gdf.to_file(OUTPUT, driver="GPKG", layer=LAYER_NAME)
print(f"✓ Exporté : {OUTPUT}  ({len(gdf)} entités)")

La couche `arrets_2026.gpkg` est prête pour les étapes suivantes

## 7. Visualisation de contrôle

Carte interactive Folium des arrêts par mode, centrée sur Rouen (zoom 11).
Chaque couche est activable/désactivable via le contrôle de légende.
La carte est sauvegardée en HTML dans `data/` pour consultation hors Jupyter.

In [ ]:
import folium
from folium.plugins import MarkerCluster

# Reprojection en WGS84 pour Folium
gdf_wgs = gdf.to_crs("EPSG:4326")

# Couleurs par mode
couleurs = {
    "Tram":  "#2ca02c",
    "TEOR":  "#1f77b4",
    "Train": "#d62728",
    "Bus":   "#7f7f7f",
    "BAC":   "#17becf",
}

m = folium.Map(
    location=[49.44, 1.09],  # centre Rouen
    zoom_start=11,
    tiles="CartoDB positron"
)

for mode, groupe in gdf_wgs.groupby("mode"):
    fg = folium.FeatureGroup(name=mode)
    for _, r in groupe.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=4,
            color=couleurs.get(mode, "#333"),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"{r['nom']} — {r['mode']}" + (f" — {r['lignes']}" if r['lignes'] else ""),
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl().add_to(m)
m.save("../data/arrets_2026_carte.html")
print("✓ Carte sauvegardée")